In [12]:
from pathlib import Path


DATA_DIR = Path("V2Data")
print(DATA_DIR.absolute())

/home/mukeshc/PP/RPS/V2Data


In [13]:
from pathlib import Path
import pandas as pd

In [ ]:
# Dataset creation

from datetime import datetime, timedelta
import numpy as np
import pandas as pd


np.random.seed(42)

START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 12, 31)

# Restaurant operating hours
OPEN_HOUR = 8
CLOSE_HOUR = 22

# -----------------------------
# HELPERS
# -----------------------------
def build_hourly_timestamps(start_date, end_date, open_hour=8, close_hour=22):
    ts = []
    current = start_date
    while current <= end_date:
        for h in range(open_hour, close_hour + 1):
            ts.append(datetime(current.year, current.month, current.day, h))
        current += timedelta(days=1)
    return ts

def get_season(month):
    if month in [3, 4, 5]:
        return "summer"
    elif month in [6, 7, 8, 9]:
        return "monsoon"
    elif month in [10, 11]:
        return "festive"
    else:
        return "winter"

def get_service_period(hour):
    if 8 <= hour <= 11:
        return "breakfast"
    elif 12 <= hour <= 15:
        return "lunch_dinner"
    elif 16 <= hour <= 18:
        return "evening"
    elif 19 <= hour <= 22:
        return "lunch_dinner"
    return "all_day"

# EXTERNAL FEATURES
def gen_external(ts_list):
    holiday_map = {
        "2025-01-01": "New Year",
        "2025-01-26": "Republic Day",
        "2025-08-15": "Independence Day",
        "2025-10-02": "Gandhi Jayanti",
        "2025-12-25": "Christmas",
    }

    festive_dates = {
        "2025-03-14": "Holi",
        "2025-08-27": "Ganesh Festival",
        "2025-10-20": "Diwali Rush",
        "2025-11-05": "Wedding Season Event",
    }

    rows = []
    for t in ts_list:
        season = get_season(t.month)
        is_weekend = int(t.weekday() >= 5)

        # Season-aware temperature
        season_temp_base = {
            "summer": 33,
            "monsoon": 28,
            "festive": 29,
            "winter": 22,
        }[season]

        hour_heat = 3 if 12 <= t.hour <= 16 else 0
        temp_c = np.random.normal(season_temp_base + hour_heat, 2.5)

        # Season-aware rain
        if season == "monsoon":
            rain_mm = np.random.choice([0, 2, 5, 10, 20, 35], p=[0.30, 0.18, 0.18, 0.16, 0.12, 0.06])
        else:
            rain_mm = np.random.choice([0, 0, 0, 1, 3, 7], p=[0.55, 0.20, 0.10, 0.07, 0.05, 0.03])

        date_str = t.strftime("%Y-%m-%d")
        holiday_name = holiday_map.get(date_str, "")
        holiday_flag = int(bool(holiday_name))

        event_name = festive_dates.get(date_str, "")
        if not event_name and np.random.rand() < 0.03:
            event_name = np.random.choice([
                "Corporate Lunch Booking",
                "College Event Crowd",
                "Cricket Match",
                "Local Fair",
                "Birthday Party"
            ])
        event_flag = int(bool(event_name))

        rows.append({
            "timestamp": t,
            "holiday_flag": holiday_flag,
            "event_flag": event_flag,
            "holiday_name": holiday_name,
            "event_name": event_name,
            "temp_c": round(float(temp_c), 1),
            "rain_mm": float(rain_mm),
            "is_weekend": is_weekend,
            "season": season
        })

    return pd.DataFrame(rows)

# SALES GENERATION
def gen_sales(df_ext):
    rows = []

    for _, row in df_ext.iterrows():
        ts = row["timestamp"]
        hour = ts.hour
        season = row["season"]

        # Base hourly traffic shape
        if 8 <= hour <= 11:
            base = 14   # breakfast
        elif 12 <= hour <= 15:
            base = 30   # lunch
        elif 16 <= hour <= 18:
            base = 18   # evening snacks
        elif 19 <= hour <= 22:
            base = 34   # dinner
        else:
            base = 8

        # Weekend / holiday / event effects
        weekend_boost = 10 if row["is_weekend"] else 0
        holiday_boost = 16 if row["holiday_flag"] else 0
        event_boost = 14 if row["event_flag"] else 0

        # Rain effect is nuanced:
        # lunch/dinner traffic may dip, tea/snack hours may rise
        if row["rain_mm"] >= 10:
            if 16 <= hour <= 19:
                rain_effect = 8
            elif 8 <= hour <= 11:
                rain_effect = 3
            else:
                rain_effect = -5
        elif row["rain_mm"] > 0:
            rain_effect = 2 if 16 <= hour <= 19 else -1
        else:
            rain_effect = 0

        # Seasonal effects
        season_effect = {
            "summer": -1 if 12 <= hour <= 15 else 1,
            "monsoon": 2 if 16 <= hour <= 19 else 0,
            "festive": 5 if 19 <= hour <= 22 else 2,
            "winter": 3 if 19 <= hour <= 22 else 1,
        }[season]

        promotion_flag = int(np.random.rand() < 0.08)
        promo_boost = 8 if promotion_flag else 0

        noise = np.random.normal(0, 4)

        covers = int(max(4, round(
            base + weekend_boost + holiday_boost + event_boost + rain_effect + season_effect + promo_boost + noise
        )))

        # Reservations vary by period
        if 19 <= hour <= 22:
            reservation_share = np.random.uniform(0.40, 0.65)
        elif 12 <= hour <= 15:
            reservation_share = np.random.uniform(0.30, 0.50)
        else:
            reservation_share = np.random.uniform(0.10, 0.30)
        if row["holiday_flag"]:
            reservation_share += 0.05
        if row["is_weekend"]:
            reservation_share += 0.03

        reservation_share = min(reservation_share, 0.80)

        reservations = int(round(covers * reservation_share))
        walk_ins = covers - reservations

        # Orders are usually slightly less than covers in a casual restaurant
        num_orders = max(1, int(round(covers * np.random.uniform(0.82, 0.96))))

        # Avg ticket depends on time and event
        avg_ticket = 10.0
        if 8 <= hour <= 11:
            avg_ticket = np.random.normal(7.5, 1.0)
        elif 12 <= hour <= 15:
            avg_ticket = np.random.normal(13.0, 1.8)
        elif 16 <= hour <= 18:
            avg_ticket = np.random.normal(8.5, 1.2)
        elif 19 <= hour <= 22:
            avg_ticket = np.random.normal(15.0, 2.2)

        if row["event_flag"]:
            avg_ticket += 1.5
        if promotion_flag:
            avg_ticket -= 0.8

        avg_ticket = max(4.5, round(float(avg_ticket), 2))
        gross_sales = round(covers * avg_ticket, 2)

        rows.append({
            "timestamp": ts,
            "covers": covers,
            "gross_sales": gross_sales,
            "num_orders": num_orders,
            "avg_ticket_size": avg_ticket,
            "reservations": reservations,
            "walk_ins": walk_ins,
            "promotion_flag": promotion_flag
        })

    return pd.DataFrame(rows)

# MENU SALES GENERATION
def choose_mix_by_context(hour, rain_mm, is_weekend, event_flag):
    """
    Returns category-level mix weights that sum roughly to 1.
    Covers are distributed to categories, then to menu items.
    """
    if 8 <= hour <= 11:  # breakfast
        mix = {
            "Breakfast": 0.48,
            "Drinks": 0.25,
            "Snacks": 0.15,
            "Main": 0.02,
            "Starter": 0.00,
            "Salad": 0.02,
            "Bread": 0.03,
            "Dessert": 0.05
        }
    elif 12 <= hour <= 15:  # lunch
        mix = {
            "Breakfast": 0.00,
            "Drinks": 0.12,
            "Snacks": 0.10,
            "Main": 0.48,
            "Starter": 0.10,
            "Salad": 0.05,
            "Bread": 0.10,
            "Dessert": 0.05
        }
    elif 16 <= hour <= 18:  # evening
        mix = {
            "Breakfast": 0.00,
            "Drinks": 0.28,
            "Snacks": 0.42,
            "Main": 0.10,
            "Starter": 0.06,
            "Salad": 0.02,
            "Bread": 0.00,
            "Dessert": 0.12
        }
    else:  # dinner
        mix = {
            "Breakfast": 0.00,
            "Drinks": 0.10,
            "Snacks": 0.08,
            "Main": 0.52,
            "Starter": 0.12,
            "Salad": 0.05,
            "Bread": 0.08,
            "Dessert": 0.05
        }

    # Rain boosts drinks/snacks; reduces heavy mains slightly
    if rain_mm >= 10:
        mix["Drinks"] += 0.10
        mix["Snacks"] += 0.08
        mix["Main"] = max(0.10, mix["Main"] - 0.12)

    # Weekend boosts starters, desserts, mains
    if is_weekend:
        mix["Starter"] += 0.03
        mix["Dessert"] += 0.03
        mix["Main"] += 0.04

    # Event days boost mains + starters
    if event_flag:
        mix["Main"] += 0.05
        mix["Starter"] += 0.04
        mix["Drinks"] += 0.01

    # Normalize
    total = sum(mix.values())
    return {k: v / total for k, v in mix.items()}

def gen_historical_menu_sales(df_sales, df_ext, menu_df):
    merged = df_sales.merge(
        df_ext[["timestamp", "rain_mm", "is_weekend", "event_flag"]],
        on="timestamp",
        how="left"
    )

    rows = []

    for _, row in merged.iterrows():
        ts = row["timestamp"]
        covers = int(row["covers"])
        hour = ts.hour

        mix = choose_mix_by_context(
            hour=hour,
            rain_mm=row["rain_mm"],
            is_weekend=row["is_weekend"],
            event_flag=row["event_flag"]
        )
        # Total sold menu-item units should usually be at least covers.
        # Breakfast tends to be closer to 1 item/cover, lunch and dinner higher.
        if 8 <= hour <= 11:          # breakfast
            item_factor = np.random.uniform(1.00, 1.15)
        elif 12 <= hour <= 15:       # lunch
            item_factor = np.random.uniform(1.15, 1.35)
        elif 16 <= hour <= 18:       # evening snacks
            item_factor = np.random.uniform(1.05, 1.25)
        else:                        # dinner
            item_factor = np.random.uniform(1.15, 1.35)

        total_item_qty = max(covers, int(round(covers * item_factor)))

        # Split total quantity across categories
        category_qty = {}
        allocated = 0
        categories = list(mix.keys())
        for i, cat in enumerate(categories):
            if i == len(categories) - 1:
                qty = total_item_qty - allocated
            else:
                qty = int(round(total_item_qty * mix[cat]))
                allocated += qty
            category_qty[cat] = max(0, qty)

        for category, qty in category_qty.items():
            if qty == 0:
                continue

            eligible = menu_df[menu_df["category"] == category].copy()

            # Filter by service period
            def allowed(service_period):
                return (
                    service_period == "all_day" or
                    (service_period == "breakfast" and 8 <= hour <= 11) or
                    (service_period == "evening" and 16 <= hour <= 18) or
                    (service_period == "lunch_dinner" and (12 <= hour <= 15 or 19 <= hour <= 22))
                )

            eligible = eligible[eligible["service_period"].apply(allowed)]

            if eligible.empty:
                continue

            # Item preference weights
            weights = np.ones(len(eligible), dtype=float)

            # Rainy day bias
            for idx, item in enumerate(eligible["menu_item_name"]):
                name = item.lower()
                if row["rain_mm"] >= 10:
                    if "tea" in name or "coffee" in name or "pakora" in name or "samosa" in name:
                        weights[idx] *= 2.5
                    if "biryani" in name or "pizza" in name:
                        weights[idx] *= 0.8

            weights = weights / weights.sum()

            item_counts = np.random.multinomial(qty, weights)

            for (_, item_row), item_qty in zip(eligible.iterrows(), item_counts):
                if item_qty > 0:
                    rows.append({
                        "timestamp": ts,
                        "menu_item_id": item_row["menu_item_id"],
                        "qty_sold": int(item_qty)
                    })

    return pd.DataFrame(rows)

timestamps = build_hourly_timestamps(
    start_date=START_DATE,
    end_date=END_DATE,
    open_hour=OPEN_HOUR,
    close_hour=CLOSE_HOUR
)

menu_df = pd.read_csv(DATA_DIR / "menu_items_master.csv")

external_df = gen_external(timestamps)
sales_df = gen_sales(external_df)
historical_menu_sales_df = gen_historical_menu_sales(sales_df, external_df, menu_df)

# Save outputs
external_df.to_csv(DATA_DIR / "external_features.csv", index=False)
sales_df.to_csv(DATA_DIR / "historical_sales.csv", index=False)
historical_menu_sales_df.to_csv(DATA_DIR / "historical_menu_sales.csv", index=False)

print("Generated files:")
print("-", DATA_DIR / "external_features.csv")
print("-", DATA_DIR / "historical_sales.csv")
print("-", DATA_DIR / "historical_menu_sales.csv")
print("-", DATA_DIR / "menu_item_master.csv")

print("\nSample external_features:")
print(external_df.head())

print("\nSample historical_sales:")
print(sales_df.head())

print("\nSample historical_menu_sales:")
print(historical_menu_sales_df.head(10))

In [14]:
# loading data sets
import pandas as pd
historical_sales_df = pd.read_csv(DATA_DIR / 'historical_sales.csv', parse_dates=['timestamp'])
external_features_df = pd.read_csv(DATA_DIR / 'external_features.csv', parse_dates=['timestamp'])
historical_menu_sales_df = pd.read_csv(DATA_DIR / 'historical_menu_sales.csv', parse_dates=['timestamp'])
menu_item_master_df = pd.read_csv(DATA_DIR / 'menu_items_master.csv')
menu_ingredients_df = pd.read_csv(DATA_DIR / 'menu_ingredients.csv')
ingredient_master_df = pd.read_csv(DATA_DIR / 'ingredient_master.csv')
staff_roles_df = pd.read_csv(DATA_DIR / 'staff_roles.csv')

covers_df = historical_sales_df.merge(external_features_df, on='timestamp', how='inner')
covers_df = covers_df.sort_values('timestamp').reset_index(drop=True)



In [ ]:
# few validations
schema_notes = []

if not historical_sales_df['timestamp'].is_unique:
    schema_notes.append('historical_sales.csv has duplicate timestamps.')
if not external_features_df['timestamp'].is_unique:
    schema_notes.append('external_features.csv has duplicate timestamps.')
if set(historical_sales_df['timestamp']) != set(external_features_df['timestamp']):
    schema_notes.append('historical_sales.csv and external_features.csv do not have the same hourly timestamps.')

missing_menu_sales_items = sorted(set(historical_menu_sales_df['menu_item_id']) - set(menu_item_master_df['menu_item_id']))
if missing_menu_sales_items:
    schema_notes.append(f'historical_menu_sales.csv contains item ids missing from menu_item_master.csv: {missing_menu_sales_items[:5]}')

missing_menu_ingredient_items = sorted(set(menu_ingredients_df['menu_item_id']) - set(menu_item_master_df['menu_item_id']))
if missing_menu_ingredient_items:
    schema_notes.append(f'menu_ingredients.csv contains item ids missing from menu_item_master.csv: {missing_menu_ingredient_items[:5]}')

missing_ingredient_ids = sorted(set(menu_ingredients_df['ingredient_id']) - set(ingredient_master_df['ingredient_id']))
if missing_ingredient_ids:
    schema_notes.append(f'menu_ingredients.csv contains ingredient ids missing from ingredient_master.csv: {missing_ingredient_ids[:5]}')

menu_stations = set(menu_item_master_df['station'])
staff_stations = set(staff_roles_df['station'])
missing_station_coverage = sorted(menu_stations - staff_stations)
if missing_station_coverage:
    print(f'Staff rules are missing for these menu production stations: {missing_station_coverage}')

hourly_menu_qty_df = historical_menu_sales_df.groupby('timestamp', as_index=False)['qty_sold'].sum()
hourly_menu_qty_df = hourly_menu_qty_df.rename(columns={'qty_sold': 'hourly_menu_qty'})
menu_cover_check_df = historical_sales_df[['timestamp', 'covers']].merge(hourly_menu_qty_df, on='timestamp', how='left')
menu_cover_check_df['hourly_menu_qty'] = menu_cover_check_df['hourly_menu_qty'].fillna(0)
menu_cover_check_df['menu_qty_per_cover'] = menu_cover_check_df['hourly_menu_qty'] / menu_cover_check_df['covers']
hours_with_fewer_items_than_covers = int((menu_cover_check_df['hourly_menu_qty'] < menu_cover_check_df['covers']).sum())
if hours_with_fewer_items_than_covers > 0:
    print(
        f'historical_menu_sales.csv has {hours_with_fewer_items_than_covers} hours where total item qty is below covers. This may be valid, but the business meaning of qty_sold should be confirmed.'
    )
    
very_low_ratio_hours = (
    menu_cover_check_df["menu_qty_per_cover"] < 0.6
).sum()

print("hours with very low items per cover:", very_low_ratio_hours)


In [ ]:
covers_df['hour'] = covers_df['timestamp'].dt.hour
covers_df['day_of_week'] = covers_df['timestamp'].dt.dayofweek
covers_df['day_of_month'] = covers_df['timestamp'].dt.day
covers_df['month'] = covers_df['timestamp'].dt.month
covers_df['week_of_year'] = covers_df['timestamp'].dt.isocalendar().week.astype(int)
covers_df['is_weekend'] = (covers_df['day_of_week'] >= 5).astype(int)
covers_df['is_month_start'] = covers_df['timestamp'].dt.is_month_start.astype(int)
covers_df['is_month_end'] = covers_df['timestamp'].dt.is_month_end.astype(int)

covers_df['hour_sin'] = np.sin(2 * np.pi * covers_df['hour'] / 24)
covers_df['hour_cos'] = np.cos(2 * np.pi * covers_df['hour'] / 24)
covers_df['dow_sin'] = np.sin(2 * np.pi * covers_df['day_of_week'] / 7)
covers_df['dow_cos'] = np.cos(2 * np.pi * covers_df['day_of_week'] / 7)

def get_service_period(hour):
    if 8 <= hour <= 11:
        return 'breakfast'
    if 12 <= hour <= 15:
        return 'lunch'
    if 16 <= hour <= 18:
        return 'evening'
    return 'dinner'

covers_df['service_period'] = covers_df['hour'].apply(get_service_period)

lag_source_df = covers_df[['timestamp', 'covers']].copy()

for lag_days, lag_name in [(1, 'covers_lag_1d'), (7, 'covers_lag_7d')]:
    lagged_df = lag_source_df.copy()
    lagged_df['timestamp'] = lagged_df['timestamp'] + pd.Timedelta(days=lag_days)
    lagged_df = lagged_df.rename(columns={'covers': lag_name})
    covers_df = covers_df.merge(lagged_df, on='timestamp', how='left')

covers_df['covers_lag_prev_open_hour'] = covers_df['covers'].shift(1)
covers_df['covers_roll_mean_3_open_hours'] = covers_df['covers'].shift(1).rolling(window=3).mean()
covers_df['covers_roll_mean_1_service_day'] = covers_df['covers'].shift(1).rolling(window=15).mean()
covers_df['covers_roll_mean_3_service_days'] = covers_df['covers'].shift(1).rolling(window=45).mean()
covers_df['covers_roll_std_3_service_days'] = covers_df['covers'].shift(1).rolling(window=45).std()

covers_df['rain_flag'] = (covers_df['rain_mm'] > 0).astype(int)
covers_df['heavy_rain_flag'] = (covers_df['rain_mm'] >= 10).astype(int)
covers_df['hot_flag'] = (covers_df['temp_c'] >= 32).astype(int)

model_base_df = covers_df[[
    'timestamp', 'covers', 'reservations', 'promotion_flag',
    'holiday_flag', 'event_flag', 'temp_c', 'rain_mm', 'rain_flag', 'heavy_rain_flag', 'hot_flag',
    'hour', 'day_of_week', 'day_of_month', 'month', 'week_of_year', 'is_weekend',
    'is_month_start', 'is_month_end', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'season', 'service_period',
    'covers_lag_prev_open_hour', 'covers_lag_1d', 'covers_lag_7d',
    'covers_roll_mean_3_open_hours', 'covers_roll_mean_1_service_day',
    'covers_roll_mean_3_service_days', 'covers_roll_std_3_service_days'
]].copy()

model_base_df = pd.get_dummies(model_base_df, columns=['season', 'service_period'], dtype=int)
model_df = model_base_df.dropna().reset_index(drop=True)

print('Merged covers dataset shape:', covers_df.shape)
print('Model-ready covers dataset shape:', model_df.shape)
model_df.head()

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor


# -----------------------------
# CONFIG
# -----------------------------
TARGET_COLUMN = "covers"
HOLDOUT_DAYS = 7
OPERATING_HOURS_PER_DAY = 15

HOLDOUT_HOURS = HOLDOUT_DAYS * OPERATING_HOURS_PER_DAY

model_df = covers_df.copy()
# -----------------------------
# FEATURES
# -----------------------------
feature_columns = [c for c in model_df.columns if c not in ["timestamp", TARGET_COLUMN]]


# -----------------------------
# SPLIT
# -----------------------------
train_df = model_df.iloc[:-HOLDOUT_HOURS].copy()
test_df = model_df.iloc[-HOLDOUT_HOURS:].copy()

X_train = train_df[feature_columns]
y_train = train_df[TARGET_COLUMN]

X_test = test_df[feature_columns]
y_test = test_df[TARGET_COLUMN]


# -----------------------------
# BASELINE
# -----------------------------
baseline_pred = test_df["covers_lag_7d"].fillna(y_train.median())


# -----------------------------
# LOG TRANSFORM TARGET
# -----------------------------
y_train_log = np.log1p(y_train)


# -----------------------------
# HYPERPARAMETER SEARCH
# -----------------------------
param_grid = {
    "n_estimators": [250, 350, 450],
    "max_depth": [4, 5, 6],
    "learning_rate": [0.03, 0.05, 0.07],
    "subsample": [0.8, 0.9],
    "colsample_bytree": [0.8, 0.9],
    "min_child_weight": [1, 3],
    "gamma": [0, 0.1],
}

tscv = TimeSeriesSplit(n_splits=3)

search = RandomizedSearchCV(
    XGBRegressor(objective="reg:squarederror", eval_metric="mae", random_state=42),
    param_distributions=param_grid,
    n_iter=15,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    verbose=1,
    n_jobs=1,
)

search.fit(X_train, y_train_log)

best_model = search.best_estimator_


# -----------------------------
# FINAL FIT
# -----------------------------
best_model.fit(X_train, y_train_log)


# -----------------------------
# PREDICT
# -----------------------------
xgb_pred_log = best_model.predict(X_test)
xgb_pred = np.expm1(xgb_pred_log)

xgb_pred = pd.Series(xgb_pred, index=X_test.index).clip(lower=0)


# -----------------------------
# ENSEMBLE
# -----------------------------
final_pred = 0.85 * xgb_pred + 0.15 * baseline_pred


# -----------------------------
# METRICS
# -----------------------------
def evaluate(y_true, y_pred, name):

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

    return {
        "model": name,
        "mae": round(mae, 3),
        "rmse": round(rmse, 3),
        "wape": round(wape, 3),
    }


metrics_df = pd.DataFrame(
    [
        evaluate(y_test, baseline_pred, "baseline"),
        evaluate(y_test, xgb_pred, "xgboost_optimized"),
        evaluate(y_test, final_pred, "ensemble"),
    ]
)

print(metrics_df)


In [ ]:
# -----------------------------
# FORECAST TABLE
# -----------------------------
forecast_df = test_df[["timestamp", "covers"]].copy()

forecast_df = forecast_df.rename(columns={"covers": "actual_covers"})

forecast_df["baseline_pred"] = baseline_pred.values
forecast_df["xgb_pred"] = xgb_pred.values
forecast_df["final_pred"] = final_pred.values

forecast_df["abs_error"] = (
    forecast_df["actual_covers"] - forecast_df["final_pred"]
).abs()

# print(forecast_df.head(20))


# # -----------------------------
# # PRODUCTION INPUT
# # -----------------------------
# production_forecast_df = forecast_df[["timestamp", "final_pred"]].rename(
#     columns={"final_pred": "predicted_covers"}
# )

# print(production_forecast_df.head())


# -----------------------------
# FEATURE IMPORTANCE
# -----------------------------
importance_df = pd.DataFrame(
    {"feature": feature_columns, "importance": best_model.feature_importances_}
).sort_values("importance", ascending=False)

print(importance_df.head(15))

In [36]:
def get_historical_pattern(df, scenario_name, condition):

    subset = df[condition].copy()

    if len(subset) < 50:
        print(f"warning: low samples for {scenario_name}")

    pattern = (
        subset
        .groupby("hour")["covers"]
        .mean()
        .reset_index()
    )

    pattern["scenario"] = scenario_name

    pattern = pattern.rename(
        columns={"covers": "historical_avg_covers"}
    )

    return pattern

historical_patterns = []

historical_patterns.append(
    get_historical_pattern(
        covers_df,
        "weekend",
        covers_df["is_weekend"] == 1
    )
)

historical_patterns.append(
    get_historical_pattern(
        covers_df,
        "heavy_rain",
        covers_df["rain_mm"] >= 10
    )
)

historical_patterns.append(
    get_historical_pattern(
        covers_df,
        "holiday",
        covers_df["holiday_flag"] == 1
    )
)

historical_patterns.append(
    get_historical_pattern(
        covers_df,
        "event",
        covers_df["event_flag"] == 1
    )
)

historical_patterns.append(
    get_historical_pattern(
        covers_df,
        "promotion",
        covers_df["promotion_flag"] == 1
    )
)

historical_patterns.append(
    get_historical_pattern(
        covers_df,
        "hot_weather",
        covers_df["hot_flag"] == 1
    )
)

historical_patterns_df = pd.concat(historical_patterns)

In [ ]:
def create_scenario_df(
    timestamps,
    rain_mm=0,
    temp_c=28,
    holiday_flag=0,
    event_flag=0,
    is_weekend=0,
    promotion_flag=0,
    season="summer",
):

    df = pd.DataFrame({"timestamp": timestamps})

    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["day_of_month"] = df["timestamp"].dt.day
    df["month"] = df["timestamp"].dt.month
    df["week_of_year"] = df["timestamp"].dt.isocalendar().week.astype(int)

    df["is_weekend"] = is_weekend

    df["holiday_flag"] = holiday_flag
    df["event_flag"] = event_flag
    df["promotion_flag"] = promotion_flag

    df["rain_mm"] = rain_mm
    df["temp_c"] = temp_c

    df["rain_flag"] = (df["rain_mm"] > 0).astype(int)
    df["heavy_rain_flag"] = (df["rain_mm"] >= 10).astype(int)

    df["hot_flag"] = (df["temp_c"] >= 32).astype(int)

    df["is_month_start"] = 0
    df["is_month_end"] = 0

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    df["season"] = season

    def get_service_period(hour):

        if 8 <= hour <= 11:
            return "breakfast"

        if 12 <= hour <= 15:
            return "lunch"

        if 16 <= hour <= 18:
            return "evening"

        return "dinner"

    df["service_period"] = df["hour"].apply(get_service_period)

    for col in [
        "covers_lag_prev_open_hour",
        "covers_lag_1d",
        "covers_lag_7d",
        "covers_roll_mean_3_open_hours",
        "covers_roll_mean_1_service_day",
        "covers_roll_mean_3_service_days",
        "covers_roll_std_3_service_days",
    ]:
        df[col] = train_df[col].median()

    df["reservations"] = train_df["reservations"].median()

    df = pd.get_dummies(df, columns=["season", "service_period"], dtype=int)

    for col in feature_columns:

        if col not in df.columns:
            df[col] = 0

    return df[feature_columns]


hours = pd.date_range(start="2026-06-01 08:00", periods=15, freq="h")

scenarios = {
    "weekend": create_scenario_df(hours, is_weekend=1),
    "heavy_rain": create_scenario_df(hours, rain_mm=20),
    "holiday": create_scenario_df(hours, holiday_flag=1),
    "event": create_scenario_df(hours, event_flag=1),
    "promotion": create_scenario_df(hours, promotion_flag=1),
    "hot_weather": create_scenario_df(hours, temp_c=35),
}

scenario_predictions = []

for scenario_name, df_scenario in scenarios.items():

    pred_log = best_model.predict(df_scenario)

    pred = np.expm1(pred_log)

    temp = pd.DataFrame(
        {"hour": hours.hour, "scenario": scenario_name, "predicted_covers": pred}
    )

    scenario_predictions.append(temp)

scenario_predictions_df = pd.concat(scenario_predictions)

comparison_df = historical_patterns_df.merge(
    scenario_predictions_df, on=["scenario", "hour"], how="inner"
)

comparison_df["difference"] = (
    comparison_df["predicted_covers"] - comparison_df["historical_avg_covers"]
)

comparison_df["error_pct"] = (
    comparison_df["difference"] / comparison_df["historical_avg_covers"]
) * 100


scenario_summary = (
    comparison_df.groupby("scenario")
    .agg(
        {
            "difference": "mean",
            "error_pct": "mean",
            "predicted_covers": "mean",
            "historical_avg_covers": "mean",
        }
    )
    .reset_index()
)

print(scenario_summary)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# =====================================================================
# IMPROVED COVERS PREDICTION MODEL
# =====================================================================
# Key fixes over previous version:
# 1. Removed 'reservations' - it's derived FROM covers (target leakage)
# 2. Added interaction features (weekend × hour, holiday × service_period, etc.)
# 3. Scenario-aware lag/rolling features using conditional medians
# 4. Better hyperparameter search range
# =====================================================================

TARGET_COLUMN = "covers"
HOLDOUT_DAYS = 7
OPERATING_HOURS_PER_DAY = 15
HOLDOUT_HOURS = HOLDOUT_DAYS * OPERATING_HOURS_PER_DAY

# -----------------------------
# REBUILD FEATURE SET (fix leakage & add interactions)
# -----------------------------
model_v2_df = covers_df.copy()
model_v2_df = model_v2_df.sort_values("timestamp").reset_index(drop=True)

# Calendar features
model_v2_df["hour"] = model_v2_df["timestamp"].dt.hour
model_v2_df["day_of_week"] = model_v2_df["timestamp"].dt.dayofweek
model_v2_df["day_of_month"] = model_v2_df["timestamp"].dt.day
model_v2_df["month"] = model_v2_df["timestamp"].dt.month
model_v2_df["week_of_year"] = model_v2_df["timestamp"].dt.isocalendar().week.astype(int)
model_v2_df["is_weekend"] = (model_v2_df["day_of_week"] >= 5).astype(int)
model_v2_df["is_month_start"] = model_v2_df["timestamp"].dt.is_month_start.astype(int)
model_v2_df["is_month_end"] = model_v2_df["timestamp"].dt.is_month_end.astype(int)

# Cyclic encodings
model_v2_df["hour_sin"] = np.sin(2 * np.pi * model_v2_df["hour"] / 24)
model_v2_df["hour_cos"] = np.cos(2 * np.pi * model_v2_df["hour"] / 24)
model_v2_df["dow_sin"] = np.sin(2 * np.pi * model_v2_df["day_of_week"] / 7)
model_v2_df["dow_cos"] = np.cos(2 * np.pi * model_v2_df["day_of_week"] / 7)
model_v2_df["month_sin"] = np.sin(2 * np.pi * model_v2_df["month"] / 12)
model_v2_df["month_cos"] = np.cos(2 * np.pi * model_v2_df["month"] / 12)

# Derived weather flags
model_v2_df["rain_flag"] = (model_v2_df["rain_mm"] > 0).astype(int)
model_v2_df["heavy_rain_flag"] = (model_v2_df["rain_mm"] >= 10).astype(int)
model_v2_df["hot_flag"] = (model_v2_df["temp_c"] >= 32).astype(int)

# Service period
def get_service_period(hour):
    if 8 <= hour <= 11:
        return "breakfast"
    if 12 <= hour <= 15:
        return "lunch"
    if 16 <= hour <= 18:
        return "evening"
    return "dinner"

model_v2_df["service_period"] = model_v2_df["hour"].apply(get_service_period)

# ---- INTERACTION FEATURES ----
# These help the model learn scenario-specific hour patterns
# model_v2_df["weekend_x_hour"] = model_v2_df["is_weekend"] * model_v2_df["hour"]
# model_v2_df["holiday_x_hour"] = model_v2_df["holiday_flag"] * model_v2_df["hour"]
# model_v2_df["event_x_hour"] = model_v2_df["event_flag"] * model_v2_df["hour"]
# model_v2_df["promo_x_hour"] = model_v2_df["promotion_flag"] * model_v2_df["hour"]
# model_v2_df["rain_x_hour"] = model_v2_df["rain_mm"] * model_v2_df["hour"]

# Weekend + event/holiday combos
model_v2_df["weekend_x_event"] = model_v2_df["is_weekend"] * model_v2_df["event_flag"]
model_v2_df["weekend_x_holiday"] = model_v2_df["is_weekend"] * model_v2_df["holiday_flag"]
model_v2_df["weekend_x_promo"] = model_v2_df["is_weekend"] * model_v2_df["promotion_flag"]

# Heavy rain during specific periods
model_v2_df["heavy_rain_x_evening"] = model_v2_df["heavy_rain_flag"] * (model_v2_df["hour"].between(16, 19)).astype(int)
model_v2_df["heavy_rain_x_lunch"] = model_v2_df["heavy_rain_flag"] * (model_v2_df["hour"].between(12, 15)).astype(int)

# ---- LAG FEATURES ----
lag_source = model_v2_df[["timestamp", "covers"]].copy()

for lag_days, lag_name in [(1, "covers_lag_1d"), (7, "covers_lag_7d")]:
    lagged = lag_source.copy()
    lagged["timestamp"] = lagged["timestamp"] + pd.Timedelta(days=lag_days)
    lagged = lagged.rename(columns={"covers": lag_name})
    model_v2_df = model_v2_df.merge(lagged, on="timestamp", how="left")

model_v2_df["covers_lag_prev_hour"] = model_v2_df["covers"].shift(1)
model_v2_df["covers_roll_mean_3h"] = model_v2_df["covers"].shift(1).rolling(window=3).mean()
model_v2_df["covers_roll_mean_1d"] = model_v2_df["covers"].shift(1).rolling(window=15).mean()
model_v2_df["covers_roll_mean_3d"] = model_v2_df["covers"].shift(1).rolling(window=45).mean()
model_v2_df["covers_roll_std_3d"] = model_v2_df["covers"].shift(1).rolling(window=45).std()

# ---- ONE-HOT ENCODE categorical columns ----
model_v2_df = pd.get_dummies(model_v2_df, columns=["season", "service_period"], dtype=int)

# ---- DEFINE FEATURE COLUMNS ----
# Explicitly EXCLUDE target + timestamp + reservations (leakage!) + walk_ins + other target-derived columns
exclude_cols = {
    "timestamp", "covers", "reservations", "walk_ins", "num_orders",
    "gross_sales", "avg_ticket_size", "holiday_name", "event_name",
    "rain_bin", "hot",  # any analysis columns
}
feature_cols_v2 = [c for c in model_v2_df.columns if c not in exclude_cols]

# ---- DROP NaN rows (from lag/rolling) ----
model_ready_df = model_v2_df.dropna(subset=feature_cols_v2).reset_index(drop=True)

print(f"Feature count: {len(feature_cols_v2)}")
print(f"Model-ready rows: {len(model_ready_df)}")

# -----------------------------
# TRAIN/TEST SPLIT
# -----------------------------
train_v2 = model_ready_df.iloc[:-HOLDOUT_HOURS].copy()
test_v2 = model_ready_df.iloc[-HOLDOUT_HOURS:].copy()

X_train_v2 = train_v2[feature_cols_v2]
y_train_v2 = train_v2[TARGET_COLUMN]
X_test_v2 = test_v2[feature_cols_v2]
y_test_v2 = test_v2[TARGET_COLUMN]

# -----------------------------
# BASELINE
# -----------------------------
baseline_pred_v2 = test_v2["covers_lag_7d"].fillna(y_train_v2.median())

# -----------------------------
# LOG TRANSFORM
# -----------------------------
y_train_log_v2 = np.log1p(y_train_v2)

# -----------------------------
# HYPERPARAMETER SEARCH
# -----------------------------
param_grid_v2 = {
    "n_estimators": [300, 500, 700],
    "max_depth": [4, 5, 6, 7],
    "learning_rate": [0.02, 0.03, 0.05],
    "subsample": [0.75, 0.85, 0.95],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.05, 0.1],
    "reg_alpha": [0, 0.1, 0.5],
    "reg_lambda": [1, 2, 5],
}

tscv_v2 = TimeSeriesSplit(n_splits=5)

search_v2 = RandomizedSearchCV(
    XGBRegressor(objective="reg:squarederror", eval_metric="mae", random_state=42),
    param_distributions=param_grid_v2,
    n_iter=30,
    scoring="neg_mean_absolute_error",
    cv=tscv_v2,
    verbose=1,
    n_jobs=-1,
    random_state=42,
)

search_v2.fit(X_train_v2, y_train_log_v2)
best_model_v2 = search_v2.best_estimator_

print(f"\nBest params: {search_v2.best_params_}")

# -----------------------------
# PREDICT
# -----------------------------
xgb_pred_log_v2 = best_model_v2.predict(X_test_v2)
xgb_pred_v2 = np.expm1(xgb_pred_log_v2)
xgb_pred_v2 = pd.Series(xgb_pred_v2, index=X_test_v2.index).clip(lower=0)

# Ensemble
ensemble_pred_v2 = 0.85 * xgb_pred_v2 + 0.15 * baseline_pred_v2

# -----------------------------
# METRICS
# -----------------------------
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))
    return {"model": name, "mae": round(mae, 3), "rmse": round(rmse, 3), "wape": round(wape, 3)}

metrics_v2 = pd.DataFrame([
    evaluate(y_test_v2, baseline_pred_v2, "baseline"),
    evaluate(y_test_v2, xgb_pred_v2, "xgboost_v2"),
    evaluate(y_test_v2, ensemble_pred_v2, "ensemble_v2"),
])
print("\n" + metrics_v2.to_string(index=False))

# -----------------------------
# FEATURE IMPORTANCE (top 20)
# -----------------------------
importance_v2 = pd.DataFrame({
    "feature": feature_cols_v2,
    "importance": best_model_v2.feature_importances_,
}).sort_values("importance", ascending=False)
print("\nTop 20 features:")
print(importance_v2.head(20).to_string(index=False))

# Keep references for scenario cell
_train_v2 = train_v2
_feature_cols_v2 = feature_cols_v2
_best_model_v2 = best_model_v2
_covers_df = covers_df


In [ ]:
import numpy as np
import pandas as pd

# =====================================================================
# SCENARIO EVALUATION WITH MULTI-CONSTRAINT SUPPORT
# =====================================================================
# Key improvements:
# 1. Scenario-aware lag features (uses conditional medians from matching history)
# 2. Multi-constraint scenarios (e.g., rainy weekend holiday)
# 3. Proper hour-level comparison against historical patterns
# 4. Summary + hourly detail views
# =====================================================================

def get_scenario_aware_lags(covers_df, condition_mask, train_end_ts):
    """
    Compute lag/rolling medians from historical data that matches the scenario condition.
    Falls back to global medians if too few matching rows.
    """
    subset = covers_df[
        (condition_mask) & (covers_df["timestamp"] <= train_end_ts)
    ].copy()

    if len(subset) < 30:
        subset = covers_df[covers_df["timestamp"] <= train_end_ts].copy()

    subset = subset.sort_values("timestamp").reset_index(drop=True)
    subset["covers_lag_prev_hour"] = subset["covers"].shift(1)
    subset["covers_lag_1d_proxy"] = subset["covers"].shift(1).rolling(15).mean()
    subset["covers_lag_7d_proxy"] = subset["covers"].shift(1).rolling(45).mean()
    subset["covers_roll_mean_3h"] = subset["covers"].shift(1).rolling(3).mean()
    subset["covers_roll_mean_1d"] = subset["covers"].shift(1).rolling(15).mean()
    subset["covers_roll_mean_3d"] = subset["covers"].shift(1).rolling(45).mean()
    subset["covers_roll_std_3d"] = subset["covers"].shift(1).rolling(45).std()

    return {
        "covers_lag_prev_hour": subset["covers_lag_prev_hour"].median(),
        "covers_lag_1d": subset["covers_lag_1d_proxy"].median(),
        "covers_lag_7d": subset["covers_lag_7d_proxy"].median(),
        "covers_roll_mean_3h": subset["covers_roll_mean_3h"].median(),
        "covers_roll_mean_1d": subset["covers_roll_mean_1d"].median(),
        "covers_roll_mean_3d": subset["covers_roll_mean_3d"].median(),
        "covers_roll_std_3d": subset["covers_roll_std_3d"].median(),
    }


def build_scenario_features(
    timestamps,
    feature_cols,
    covers_df,
    train_end_ts,
    # Scenario conditions
    rain_mm=0,
    temp_c=28,
    holiday_flag=0,
    event_flag=0,
    is_weekend=0,
    promotion_flag=0,
    season="monsoon",
):
    """Build a feature DataFrame for scenario prediction with context-aware lags."""

    df = pd.DataFrame({"timestamp": timestamps})

    # Calendar features
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["day_of_month"] = df["timestamp"].dt.day
    df["month"] = df["timestamp"].dt.month
    df["week_of_year"] = df["timestamp"].dt.isocalendar().week.astype(int)
    df["is_weekend"] = is_weekend
    df["is_month_start"] = 0
    df["is_month_end"] = 0

    # Cyclic
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    # Scenario flags
    df["holiday_flag"] = holiday_flag
    df["event_flag"] = event_flag
    df["promotion_flag"] = promotion_flag
    df["rain_mm"] = rain_mm
    df["temp_c"] = temp_c
    df["rain_flag"] = int(rain_mm > 0)
    df["heavy_rain_flag"] = int(rain_mm >= 10)
    df["hot_flag"] = int(temp_c >= 32)

    # Interaction features
    df["weekend_x_hour"] = df["is_weekend"] * df["hour"]
    df["holiday_x_hour"] = df["holiday_flag"] * df["hour"]
    df["event_x_hour"] = df["event_flag"] * df["hour"]
    df["promo_x_hour"] = df["promotion_flag"] * df["hour"]
    df["rain_x_hour"] = df["rain_mm"] * df["hour"]
    df["weekend_x_event"] = df["is_weekend"] * df["event_flag"]
    df["weekend_x_holiday"] = df["is_weekend"] * df["holiday_flag"]
    df["weekend_x_promo"] = df["is_weekend"] * df["promotion_flag"]
    df["heavy_rain_x_evening"] = df["heavy_rain_flag"] * (df["hour"].between(16, 19)).astype(int)
    df["heavy_rain_x_lunch"] = df["heavy_rain_flag"] * (df["hour"].between(12, 15)).astype(int)

    # Season & service period
    df["season"] = season

    def get_service_period(hour):
        if 8 <= hour <= 11:
            return "breakfast"
        if 12 <= hour <= 15:
            return "lunch"
        if 16 <= hour <= 18:
            return "evening"
        return "dinner"

    df["service_period"] = df["hour"].apply(get_service_period)

    # One-hot
    df = pd.get_dummies(df, columns=["season", "service_period"], dtype=int)

    # -- Scenario-aware lag features --
    # Build condition mask on historical data matching this scenario
    mask = pd.Series(True, index=covers_df.index)
    if is_weekend:
        mask &= covers_df["is_weekend"] == 1
    if holiday_flag:
        mask &= covers_df["holiday_flag"] == 1
    if event_flag:
        mask &= covers_df["event_flag"] == 1
    if promotion_flag:
        mask &= covers_df["promotion_flag"] == 1
    if rain_mm >= 10:
        mask &= covers_df["rain_mm"] >= 10
    if temp_c >= 32:
        mask &= covers_df["temp_c"] >= 32

    lags = get_scenario_aware_lags(covers_df, mask, train_end_ts)
    for col, val in lags.items():
        df[col] = val

    # Ensure all feature columns exist
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0

    return df[feature_cols]


# =====================================================================
# DEFINE SCENARIOS (single + multi-constraint)
# =====================================================================

train_end_ts = _train_v2["timestamp"].max()

# Prediction timestamps: a sample day
scenario_hours = pd.date_range(start="2026-06-06 08:00", periods=15, freq="h")  # a Saturday

scenarios = {
    # --- Single-factor scenarios ---
    "normal_weekday": {
        "rain_mm": 0, "temp_c": 28, "holiday_flag": 0, "event_flag": 0,
        "is_weekend": 0, "promotion_flag": 0, "season": "monsoon",
    },
    "weekend": {
        "rain_mm": 0, "temp_c": 28, "holiday_flag": 0, "event_flag": 0,
        "is_weekend": 1, "promotion_flag": 0, "season": "monsoon",
    },
    "holiday": {
        "rain_mm": 0, "temp_c": 28, "holiday_flag": 1, "event_flag": 0,
        "is_weekend": 0, "promotion_flag": 0, "season": "monsoon",
    },
    "event": {
        "rain_mm": 0, "temp_c": 28, "holiday_flag": 0, "event_flag": 1,
        "is_weekend": 0, "promotion_flag": 0, "season": "monsoon",
    },
    "promotion": {
        "rain_mm": 0, "temp_c": 28, "holiday_flag": 0, "event_flag": 0,
        "is_weekend": 0, "promotion_flag": 1, "season": "monsoon",
    },
    "heavy_rain": {
        "rain_mm": 20, "temp_c": 26, "holiday_flag": 0, "event_flag": 0,
        "is_weekend": 0, "promotion_flag": 0, "season": "monsoon",
    },
    "hot_weather": {
        "rain_mm": 0, "temp_c": 38, "holiday_flag": 0, "event_flag": 0,
        "is_weekend": 0, "promotion_flag": 0, "season": "summer",
    },
    # --- Multi-constraint scenarios ---
    "weekend_event": {
        "rain_mm": 0, "temp_c": 28, "holiday_flag": 0, "event_flag": 1,
        "is_weekend": 1, "promotion_flag": 0, "season": "monsoon",
    },
    "holiday_weekend": {
        "rain_mm": 0, "temp_c": 28, "holiday_flag": 1, "event_flag": 0,
        "is_weekend": 1, "promotion_flag": 0, "season": "winter",
    },
    "rainy_weekend": {
        "rain_mm": 15, "temp_c": 26, "holiday_flag": 0, "event_flag": 0,
        "is_weekend": 1, "promotion_flag": 0, "season": "monsoon",
    },
    "weekend_promo": {
        "rain_mm": 0, "temp_c": 28, "holiday_flag": 0, "event_flag": 0,
        "is_weekend": 1, "promotion_flag": 1, "season": "monsoon",
    },
    "holiday_event_weekend": {
        "rain_mm": 0, "temp_c": 28, "holiday_flag": 1, "event_flag": 1,
        "is_weekend": 1, "promotion_flag": 0, "season": "festive",
    },
    "rainy_event": {
        "rain_mm": 20, "temp_c": 26, "holiday_flag": 0, "event_flag": 1,
        "is_weekend": 0, "promotion_flag": 0, "season": "monsoon",
    },
}

# =====================================================================
# PREDICT & BUILD HISTORICAL COMPARISON
# =====================================================================

def get_historical_avg(covers_df, condition_mask, train_end_ts):
    """Get hourly average covers from historical data matching condition."""
    subset = covers_df[
        (condition_mask) & (covers_df["timestamp"] <= train_end_ts)
    ].copy()
    subset["hour"] = subset["timestamp"].dt.hour
    return subset.groupby("hour")["covers"].mean()


all_scenario_results = []

for name, params in scenarios.items():
    # Build features
    feat_df = build_scenario_features(
        timestamps=scenario_hours,
        feature_cols=_feature_cols_v2,
        covers_df=_covers_df,
        train_end_ts=train_end_ts,
        **params,
    )

    # Predict
    pred_log = _best_model_v2.predict(feat_df)
    pred = np.expm1(pred_log).clip(min=0)

    # Historical comparison mask
    hist_mask = pd.Series(True, index=_covers_df.index)
    if params["is_weekend"]:
        hist_mask &= _covers_df["is_weekend"] == 1
    if params["holiday_flag"]:
        hist_mask &= _covers_df["holiday_flag"] == 1
    if params["event_flag"]:
        hist_mask &= _covers_df["event_flag"] == 1
    if params["promotion_flag"]:
        hist_mask &= _covers_df["promotion_flag"] == 1
    if params["rain_mm"] >= 10:
        hist_mask &= _covers_df["rain_mm"] >= 10
    if params["temp_c"] >= 32:
        hist_mask &= _covers_df["temp_c"] >= 32

    # For normal_weekday, use non-special days
    if name == "normal_weekday":
        hist_mask = (
            (_covers_df["is_weekend"] == 0) &
            (_covers_df["holiday_flag"] == 0) &
            (_covers_df["event_flag"] == 0) &
            (_covers_df["promotion_flag"] == 0)
        )

    hist_avg = get_historical_avg(_covers_df, hist_mask, train_end_ts)

    for i, ts in enumerate(scenario_hours):
        h = ts.hour
        hist_val = hist_avg.get(h, np.nan)
        all_scenario_results.append({
            "scenario": name,
            "hour": h,
            "predicted_covers": round(pred[i], 1),
            "historical_avg": round(hist_val, 1) if not np.isnan(hist_val) else np.nan,
        })

results_df = pd.DataFrame(all_scenario_results)
results_df["difference"] = results_df["predicted_covers"] - results_df["historical_avg"]
results_df["error_pct"] = (results_df["difference"] / results_df["historical_avg"]) * 100

# =====================================================================
# SUMMARY TABLE
# =====================================================================
summary = (
    results_df
    .groupby("scenario")
    .agg(
        predicted_avg=("predicted_covers", "mean"),
        historical_avg=("historical_avg", "mean"),
        mean_diff=("difference", "mean"),
        mean_error_pct=("error_pct", "mean"),
        max_abs_error_pct=("error_pct", lambda x: x.abs().max()),
    )
    .round(2)
    .reset_index()
)

# Reorder to show single-factor first, then multi-constraint
single_scenarios = ["normal_weekday", "weekend", "holiday", "event", "promotion", "heavy_rain", "hot_weather"]
multi_scenarios = [s for s in scenarios.keys() if s not in single_scenarios]

summary["sort_key"] = summary["scenario"].apply(
    lambda s: single_scenarios.index(s) if s in single_scenarios else len(single_scenarios) + multi_scenarios.index(s)
)
summary = summary.sort_values("sort_key").drop(columns=["sort_key"]).reset_index(drop=True)

print("=" * 90)
print("SCENARIO PREDICTION SUMMARY (predicted vs historical average)")
print("=" * 90)
print(summary.to_string(index=False))

# =====================================================================
# HOURLY DETAIL FOR KEY SCENARIOS
# =====================================================================
print("\n")
print("=" * 90)
print("HOURLY DETAIL - Selected Scenarios")
print("=" * 90)

for scenario_name in ["normal_weekday", "weekend", "holiday", "event", "weekend_event", "holiday_event_weekend"]:
    if scenario_name in scenarios:
        detail = results_df[results_df["scenario"] == scenario_name][
            ["hour", "predicted_covers", "historical_avg", "difference", "error_pct"]
        ].copy()
        detail["error_pct"] = detail["error_pct"].round(1)
        print(f"\n--- {scenario_name} ---")
        print(detail.to_string(index=False))

# =====================================================================
# QUALITY CHECK: Direction accuracy
# =====================================================================
print("\n")
print("=" * 90)
print("QUALITY CHECK: Does the model predict higher covers for boost scenarios?")
print("=" * 90)

normal_avg = summary[summary["scenario"] == "normal_weekday"]["predicted_avg"].values[0]

for _, row in summary.iterrows():
    if row["scenario"] == "normal_weekday":
        continue
    direction = "↑" if row["predicted_avg"] > normal_avg else "↓"
    delta = row["predicted_avg"] - normal_avg
    print(f"  {row['scenario']:30s}  {direction}  {delta:+.1f} covers vs normal  (pred={row['predicted_avg']:.1f}, hist={row['historical_avg']:.1f})")


In [17]:
# current final

import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


class RestaurantForecaster:
    """
    Single-model restaurant forecaster using XGBoost.
    Includes:
    - feature engineering
    - point forecast
    - quantile forecasts for uncertainty
    - scenario testing support
    """

    def __init__(self, feature_cols=None):
        self.feature_cols = feature_cols or []
        self.model = None
        self.quantile_models = {}

    def build_enhanced_features(self, df, covers_df=None, train_end_ts=None):
        df = df.copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.sort_values("timestamp").reset_index(drop=True)

        # ----------------------------
        # TIME FEATURES
        # ----------------------------
        df["hour"] = df["timestamp"].dt.hour
        df["day_of_week"] = df["timestamp"].dt.dayofweek
        df["day_of_month"] = df["timestamp"].dt.day
        df["month"] = df["timestamp"].dt.month
        df["week_of_year"] = df["timestamp"].dt.isocalendar().week.astype(int)

        if "is_weekend" not in df.columns:
            df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

        df["is_friday"] = (df["day_of_week"] == 4).astype(int)
        df["is_month_start"] = df["timestamp"].dt.is_month_start.astype(int)
        df["is_month_end"] = df["timestamp"].dt.is_month_end.astype(int)

        # Peaks
        df["is_breakfast_peak"] = df["hour"].between(9, 11).astype(int)
        df["is_lunch_peak"] = df["hour"].between(12, 14).astype(int)
        df["is_dinner_peak"] = df["hour"].between(19, 21).astype(int)
        df["is_any_peak"] = (
            df["is_breakfast_peak"] | df["is_lunch_peak"] | df["is_dinner_peak"]
        ).astype(int)

        df["hours_to_lunch"] = (13 - df["hour"]).clip(-12, 12)
        df["hours_to_dinner"] = (20 - df["hour"]).clip(-12, 12)

        # Cyclic
        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
        df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
        df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

        # ----------------------------
        # WEATHER FEATURES
        # ----------------------------
        if "rain_mm" in df.columns:
            df["rain_flag"] = (df["rain_mm"] > 0).astype(int)
            df["moderate_rain_flag"] = df["rain_mm"].between(5, 15).astype(int)
            df["heavy_rain_flag"] = (df["rain_mm"] >= 15).astype(int)
            df["rain_category"] = pd.cut(
                df["rain_mm"],
                bins=[-np.inf, 0, 5, 15, np.inf],
                labels=["none", "light", "moderate", "heavy"]
            )

            df["is_weekend_dinner"] = df["is_weekend"] * df["is_dinner_peak"]

            df["rain_evening_effect"] = df["heavy_rain_flag"] * df["hour"].between(
                16, 19
            ).astype(int)

            df["rain_dinner_penalty"] = df["heavy_rain_flag"] * df["hour"].between(
                20, 22
            ).astype(int)

        if "temp_c" in df.columns:
            df["temp_comfortable"] = df["temp_c"].between(22, 30).astype(int)
            df["temp_hot"] = (df["temp_c"] > 35).astype(int)
            df["temp_very_hot"] = (df["temp_c"] > 38).astype(int)
            df["temp_cold"] = (df["temp_c"] < 18).astype(int)

        # ----------------------------
        # SERVICE PERIOD
        # ----------------------------
        def get_service_period(hour):
            if hour < 8:
                return "pre_service"
            elif 8 <= hour <= 11:
                return "breakfast"
            elif 12 <= hour <= 15:
                return "lunch"
            elif 16 <= hour <= 18:
                return "afternoon"
            elif 19 <= hour <= 22:
                return "dinner"
            else:
                return "late_night"

        df["service_period"] = df["hour"].apply(get_service_period)

        # ----------------------------
        # INTERACTIONS
        # ----------------------------
        if "holiday_flag" in df.columns:
            df["holiday_x_hour"] = df["holiday_flag"] * df["hour"]
            df["holiday_x_lunch"] = df["holiday_flag"] * df["is_lunch_peak"]
            df["holiday_x_dinner"] = df["holiday_flag"] * df["is_dinner_peak"]

        if "event_flag" in df.columns:
            df["event_x_hour"] = df["event_flag"] * df["hour"]
            df["event_x_dinner"] = df["event_flag"] * df["is_dinner_peak"]

        if "promotion_flag" in df.columns:
            df["promo_x_hour"] = df["promotion_flag"] * df["hour"]
            df["promo_x_lunch"] = df["promotion_flag"] * df["is_lunch_peak"]

        if "rain_mm" in df.columns:
            df["rain_x_hour"] = df["rain_mm"] * df["hour"]
            df["rain_x_lunch"] = df["rain_mm"] * df["is_lunch_peak"]
            df["rain_x_dinner"] = df["rain_mm"] * df["is_dinner_peak"]

        if "temp_c" in df.columns:
            df["temp_x_hour"] = df["temp_c"] * df["hour"]
            df["hot_x_lunch"] = df["temp_hot"] * df["is_lunch_peak"]
            df["hot_x_dinner"] = df["temp_hot"] * df["is_dinner_peak"]

        # ----------------------------
        # LAGS / ROLLING
        # ----------------------------
        if covers_df is not None and train_end_ts is not None:
            lag_features = self._get_scenario_aware_lags(df, covers_df, train_end_ts)
            for col, val in lag_features.items():
                df[col] = val
        else:
            if "covers" in df.columns:
                df["covers_lag_1h"] = df["covers"].shift(1)
                df["covers_lag_1d"] = df["covers"].shift(15)
                df["covers_lag_7d"] = df["covers"].shift(7 * 15)
                df["covers_lag_1w_same_hour"] = df.groupby("hour")["covers"].shift(7 * 15)
                df["covers_lag_1d_same_period"] = df.groupby("service_period")["covers"].shift(15)

                df["covers_roll_mean_3h"] = df["covers"].shift(1).rolling(3).mean()
                df["covers_roll_mean_6h"] = df["covers"].shift(1).rolling(6).mean()
                df["covers_roll_mean_1d"] = df["covers"].shift(1).rolling(15).mean()
                df["covers_roll_mean_3d"] = df["covers"].shift(1).rolling(45).mean()
                df["covers_roll_mean_7d"] = df["covers"].shift(1).rolling(105).mean()

                df["covers_roll_std_1d"] = df["covers"].shift(1).rolling(15).std()
                df["covers_roll_std_3d"] = df["covers"].shift(1).rolling(45).std()
                df["covers_roll_max_1d"] = df["covers"].shift(1).rolling(15).max()
                df["covers_roll_min_1d"] = df["covers"].shift(1).rolling(15).min()

                df["covers_volatility_3d"] = (
                    df["covers_roll_std_3d"] / (df["covers_roll_mean_3d"] + 1)
                )

        # ----------------------------
        # CATEGORICALS
        # ----------------------------
        if "season" in df.columns:
            df["season"] = pd.Categorical(
                df["season"],
                categories=["winter", "summer", "monsoon", "festive"]
            )
            df = pd.get_dummies(df, columns=["season"], prefix="season", dtype=int)

        if "service_period" in df.columns:
            df["service_period"] = pd.Categorical(
                df["service_period"],
                categories=["pre_service", "breakfast", "lunch", "afternoon", "dinner", "late_night"]
            )
            df = pd.get_dummies(df, columns=["service_period"], prefix="period", dtype=int)

        if "rain_category" in df.columns:
            df["rain_category"] = pd.Categorical(
                df["rain_category"],
                categories=["none", "light", "moderate", "heavy"]
            )
            df = pd.get_dummies(df, columns=["rain_category"], prefix="rain_cat", dtype=int)

        return df

    def _get_scenario_aware_lags(self, df, covers_df, train_end_ts):
        row = df.iloc[0]
        mask = pd.Series(True, index=covers_df.index)

        if "holiday_flag" in row and row["holiday_flag"]:
            mask &= (covers_df["holiday_flag"] == 1)
        if "event_flag" in row and row["event_flag"]:
            mask &= (covers_df["event_flag"] == 1)
        if "promotion_flag" in row and row["promotion_flag"]:
            mask &= (covers_df["promotion_flag"] == 1)
        if "rain_mm" in row and row["rain_mm"] >= 15:
            mask &= (covers_df["rain_mm"] >= 15)
        if "temp_c" in row and row["temp_c"] >= 35:
            mask &= (covers_df["temp_c"] >= 35)

        subset = covers_df[mask & (covers_df["timestamp"] <= train_end_ts)].copy()
        if len(subset) < 50:
            subset = covers_df[covers_df["timestamp"] <= train_end_ts].copy()

        subset = subset.sort_values("timestamp").reset_index(drop=True)

        return {
            "covers_lag_1h": subset["covers"].median(),
            "covers_lag_1d": subset["covers"].median(),
            "covers_lag_7d": subset["covers"].median(),
            "covers_lag_1w_same_hour": subset["covers"].median(),
            "covers_lag_1d_same_period": subset["covers"].median(),
            "covers_roll_mean_3h": subset["covers"].median(),
            "covers_roll_mean_6h": subset["covers"].median(),
            "covers_roll_mean_1d": subset["covers"].median(),
            "covers_roll_mean_3d": subset["covers"].median(),
            "covers_roll_mean_7d": subset["covers"].median(),
            "covers_roll_std_1d": subset["covers"].std(),
            "covers_roll_std_3d": subset["covers"].std(),
            "covers_roll_max_1d": subset["covers"].quantile(0.75),
            "covers_roll_min_1d": subset["covers"].quantile(0.25),
            "covers_volatility_3d": 0.15,
        }

    def train(self, X_train, y_train):
        y_train_log = np.log1p(y_train)

        self.model = XGBRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=1,
            gamma=0.1,
            reg_alpha=0.5,
            reg_lambda=5,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )
        self.model.fit(X_train, y_train_log, verbose=False)

        for q in [0.1, 0.5, 0.9]:
            self.quantile_models[f"q{int(q*100)}"] = XGBRegressor(
                n_estimators=400,
                max_depth=5,
                learning_rate=0.03,
                objective="reg:quantileerror",
                quantile_alpha=q,
                random_state=42,
                n_jobs=-1
            )
            self.quantile_models[f"q{int(q*100)}"].fit(X_train, y_train, verbose=False)

    def predict(self, X, return_uncertainty=False):
        pred_log = self.model.predict(X)
        pred = np.expm1(pred_log).clip(min=0)

        if return_uncertainty:
            q10 = self.quantile_models["q10"].predict(X).clip(min=0)
            q50 = self.quantile_models["q50"].predict(X).clip(min=0)
            q90 = self.quantile_models["q90"].predict(X).clip(min=0)
            return pred, q10, q50, q90

        return pred

    def evaluate(self, X_test, y_test):
        pred = self.predict(X_test)
        mae = mean_absolute_error(y_test, pred)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        wape = np.sum(np.abs(y_test - pred)) / np.sum(np.abs(y_test))
        return {
            "mae": mae,
            "rmse": rmse,
            "wape": wape
        }

In [9]:
def run_single_xgb_forecast(covers_df, holdout_hours=7 * 15):
    covers_df = covers_df.copy()
    covers_df["timestamp"] = pd.to_datetime(covers_df["timestamp"])
    covers_df = covers_df.sort_values("timestamp").reset_index(drop=True)

    forecaster = RestaurantForecaster()

    # Build features on full dataset first
    full_enhanced = forecaster.build_enhanced_features(covers_df.copy())

    exclude = {
        "timestamp", "covers", "reservations", "walk_ins", "num_orders",
        "gross_sales", "avg_ticket_size", "holiday_name", "event_name"
    }

    feature_cols = [c for c in full_enhanced.columns if c not in exclude]
    full_enhanced = full_enhanced.dropna(subset=feature_cols).reset_index(drop=True)

    train_df = full_enhanced.iloc[:-holdout_hours].copy()
    test_df = full_enhanced.iloc[-holdout_hours:].copy()

    X_train = train_df[feature_cols]
    y_train = train_df["covers"]
    X_test = test_df[feature_cols]
    y_test = test_df["covers"]

    forecaster.feature_cols = feature_cols
    forecaster.train(X_train, y_train)

    pred, q10, q50, q90 = forecaster.predict(X_test, return_uncertainty=True)
    metrics = forecaster.evaluate(X_test, y_test)

    print("\n" + "=" * 80)
    print("SINGLE XGBOOST FORECAST RESULTS")
    print("=" * 80)
    print(f"Features used: {len(feature_cols)}")
    print(f"Train rows: {len(train_df)}")
    print(f"Test rows: {len(test_df)}")
    print(f"MAE : {metrics['mae']:.3f}")
    print(f"RMSE: {metrics['rmse']:.3f}")
    print(f"WAPE: {metrics['wape']:.3f}")
    print("=" * 80)

    return forecaster, train_df, test_df, pred, q10, q50, q90, metrics


    print("\n" + "=" * 80)
    print("SCENARIO TESTING - SINGLE XGBOOST")
    print("=" * 80 + "\n")

    scenario_starts = {
        "normal_weekday": "2026-06-03 08:00",   # Wednesday
        "heavy_rain": "2026-06-03 08:00",
        "hot_weather": "2026-05-15 08:00",      # summer weekday
        "weekend": "2026-06-06 08:00",          # Saturday
        "holiday": "2026-12-25 08:00",
        "holiday_weekend": "2026-12-26 08:00"
    }

    scenarios = {
        "normal_weekday": {
            "rain_mm": 0, "temp_c": 28, "holiday_flag": 0, "event_flag": 0,
            "promotion_flag": 0, "season": "monsoon",
            "reservations": 18, "walk_ins": 22, "num_orders": 36,
            "gross_sales": 500.0, "avg_ticket_size": 12.0,
            "holiday_name": "", "event_name": ""
        },
        "heavy_rain": {
            "rain_mm": 20, "temp_c": 26, "holiday_flag": 0, "event_flag": 0,
            "promotion_flag": 0, "season": "monsoon",
            "reservations": 15, "walk_ins": 17, "num_orders": 30,
            "gross_sales": 420.0, "avg_ticket_size": 11.0,
            "holiday_name": "", "event_name": ""
        },
        "hot_weather": {
            "rain_mm": 0, "temp_c": 38, "holiday_flag": 0, "event_flag": 0,
            "promotion_flag": 0, "season": "summer",
            "reservations": 17, "walk_ins": 21, "num_orders": 34,
            "gross_sales": 470.0, "avg_ticket_size": 12.5,
            "holiday_name": "", "event_name": ""
        },
        "weekend": {
            "rain_mm": 0, "temp_c": 28, "holiday_flag": 0, "event_flag": 0,
            "promotion_flag": 0, "season": "monsoon",
            "reservations": 25, "walk_ins": 28, "num_orders": 46,
            "gross_sales": 700.0, "avg_ticket_size": 13.5,
            "holiday_name": "", "event_name": ""
        },
        "holiday": {
            "rain_mm": 0, "temp_c": 28, "holiday_flag": 1, "event_flag": 0,
            "promotion_flag": 0, "season": "winter",
            "reservations": 28, "walk_ins": 30, "num_orders": 52,
            "gross_sales": 820.0, "avg_ticket_size": 14.0,
            "holiday_name": "Public Holiday", "event_name": ""
        },
        "holiday_weekend": {
            "rain_mm": 0, "temp_c": 24, "holiday_flag": 1, "event_flag": 0,
            "promotion_flag": 0, "season": "winter",
            "reservations": 32, "walk_ins": 34, "num_orders": 58,
            "gross_sales": 930.0, "avg_ticket_size": 14.5,
            "holiday_name": "Public Holiday", "event_name": ""
        }
    }

    results = {}

    for name, params in scenarios.items():
        scenario_hours = pd.date_range(start=scenario_starts[name], periods=15, freq="h")
        df_scenario = pd.DataFrame({"timestamp": scenario_hours})

        for k, v in params.items():
            df_scenario[k] = v

        df_scenario = forecaster.build_enhanced_features(
            df_scenario.copy(),
            covers_df=covers_df,
            train_end_ts=train_end_ts
        )

        for col in forecaster.feature_cols:
            if col not in df_scenario.columns:
                df_scenario[col] = 0

        X_scenario = df_scenario[forecaster.feature_cols]

        pred, q10, q50, q90 = forecaster.predict(X_scenario, return_uncertainty=True)

        results[name] = {
            "predicted": pred,
            "q10": q10,
            "q50": q50,
            "q90": q90,
            "avg": float(np.mean(pred))
        }

    print(f"{'Scenario':<25} {'Avg Pred Covers':<20} {'vs Normal'}")
    print("-" * 65)

    normal_avg = results["normal_weekday"]["avg"]

    for name in scenarios:
        avg = results[name]["avg"]
        diff = avg - normal_avg
        direction = "↑" if diff > 0 else "↓"
        print(f"{name:<25} {avg:<20.1f} {direction} {diff:+.1f}")

    return results

In [11]:
import numpy as np
import pandas as pd

def test_scenarios(forecaster, covers_df, train_end_ts):
    """
    Scenario validation for the restaurant demand forecaster.

    Tests only exogenous / known-ahead inputs:
    - calendar effects
    - weather
    - promotions
    - events
    - reservations

    It avoids leakage-like fields such as:
    - gross_sales
    - num_orders
    - avg_ticket_size
    - walk_ins

    Returns
    -------
    results : dict
        Scenario-level predictions and summary stats.
    validation_df : pd.DataFrame
        Compact validation table for easy review.
    """

    print("\n" + "=" * 90)
    print("SCENARIO TESTING - SYSTEM VALIDATION")
    print("=" * 90 + "\n")

    # ----------------------------
    # Scenario timestamps
    # ----------------------------
    # Choose dates that naturally match the scenario type.
    scenario_starts = {
        "normal_weekday": "2026-06-03 08:00",   # Wednesday
        "heavy_rain_weekday": "2026-06-03 08:00",
        "hot_weekday": "2026-05-15 08:00",      # summer weekday
        "weekend": "2026-06-06 08:00",          # Saturday
        "holiday_weekday": "2026-12-25 08:00",  # holiday
        "holiday_weekend": "2026-12-26 08:00",  # synthetic holiday weekend test
        "promotion_weekday": "2026-06-03 08:00",
        "event_weekday": "2026-06-03 08:00",
        "event_weekend": "2026-06-06 08:00",
        "rain_weekend": "2026-06-06 08:00",
    }

    # ----------------------------
    # Base hourly reservation profile
    # ----------------------------
    def reservation_profile(hours, base_type="weekday"):
        vals = []
        for h in hours:
            if 8 <= h <= 11:       # breakfast
                base = 8 if base_type == "weekday" else 10
            elif 12 <= h <= 15:    # lunch
                base = 18 if base_type == "weekday" else 24
            elif 16 <= h <= 18:    # afternoon / evening snack
                base = 10 if base_type == "weekday" else 14
            else:                  # dinner
                base = 24 if base_type == "weekday" else 30
            vals.append(base)
        return vals

    # ----------------------------
    # Scenario definitions
    # ----------------------------
    scenarios = {
        "normal_weekday": {
            "rain_mm": 0,
            "temp_c": 28,
            "holiday_flag": 0,
            "event_flag": 0,
            "promotion_flag": 0,
            "season": "monsoon",
            "holiday_name": "",
            "event_name": "",
            "reservation_type": "weekday",
        },
        "heavy_rain_weekday": {
            "rain_mm": 20,
            "temp_c": 26,
            "holiday_flag": 0,
            "event_flag": 0,
            "promotion_flag": 0,
            "season": "monsoon",
            "holiday_name": "",
            "event_name": "",
            "reservation_type": "weekday",
        },
        "hot_weekday": {
            "rain_mm": 0,
            "temp_c": 38,
            "holiday_flag": 0,
            "event_flag": 0,
            "promotion_flag": 0,
            "season": "summer",
            "holiday_name": "",
            "event_name": "",
            "reservation_type": "weekday",
        },
        "weekend": {
            "rain_mm": 0,
            "temp_c": 28,
            "holiday_flag": 0,
            "event_flag": 0,
            "promotion_flag": 0,
            "season": "monsoon",
            "holiday_name": "",
            "event_name": "",
            "reservation_type": "weekend",
        },
        "holiday_weekday": {
            "rain_mm": 0,
            "temp_c": 24,
            "holiday_flag": 1,
            "event_flag": 0,
            "promotion_flag": 0,
            "season": "winter",
            "holiday_name": "Public Holiday",
            "event_name": "",
            "reservation_type": "weekend",  # holidays usually behave closer to high-demand days
        },
        "holiday_weekend": {
            "rain_mm": 0,
            "temp_c": 24,
            "holiday_flag": 1,
            "event_flag": 0,
            "promotion_flag": 0,
            "season": "winter",
            "holiday_name": "Public Holiday",
            "event_name": "",
            "reservation_type": "weekend",
        },
        "promotion_weekday": {
            "rain_mm": 0,
            "temp_c": 28,
            "holiday_flag": 0,
            "event_flag": 0,
            "promotion_flag": 1,
            "season": "monsoon",
            "holiday_name": "",
            "event_name": "",
            "reservation_type": "weekday",
        },
        "event_weekday": {
            "rain_mm": 0,
            "temp_c": 28,
            "holiday_flag": 0,
            "event_flag": 1,
            "promotion_flag": 0,
            "season": "monsoon",
            "holiday_name": "",
            "event_name": "Corporate Lunch Booking",
            "reservation_type": "weekday",
        },
        "event_weekend": {
            "rain_mm": 0,
            "temp_c": 28,
            "holiday_flag": 0,
            "event_flag": 1,
            "promotion_flag": 0,
            "season": "monsoon",
            "holiday_name": "",
            "event_name": "Birthday Party",
            "reservation_type": "weekend",
        },
        "rain_weekend": {
            "rain_mm": 18,
            "temp_c": 25,
            "holiday_flag": 0,
            "event_flag": 0,
            "promotion_flag": 0,
            "season": "monsoon",
            "holiday_name": "",
            "event_name": "",
            "reservation_type": "weekend",
        },
    }

    results = {}

    # ----------------------------
    # Run scenarios
    # ----------------------------
    for name, params in scenarios.items():
        scenario_hours = pd.date_range(
            start=scenario_starts[name],
            periods=15,
            freq="h"
        )
        df_scenario = pd.DataFrame({"timestamp": scenario_hours})

        # assign clean known-ahead inputs
        for k, v in params.items():
            if k != "reservation_type":
                df_scenario[k] = v

        # reservations vary by hour
        df_scenario["reservations"] = reservation_profile(
            df_scenario["timestamp"].dt.hour,
            base_type=params["reservation_type"]
        )

        # build features with historical context for lag-like scenario features
        df_scenario = forecaster.build_enhanced_features(
            df_scenario.copy(),
            covers_df=covers_df,
            train_end_ts=train_end_ts
        )

        # ensure all expected model columns exist
        for col in forecaster.feature_cols:
            if col not in df_scenario.columns:
                df_scenario[col] = 0

        X_scenario = df_scenario[forecaster.feature_cols]

        pred, q10, q50, q90 = forecaster.predict(
            X_scenario,
            return_uncertainty=True
        )

        results[name] = {
            "predicted": pred,
            "q10": q10,
            "q50": q50,
            "q90": q90,
            "avg": float(np.mean(pred)),
            "peak_hour_pred": float(np.max(pred)),
            "total_day_pred": float(np.sum(pred)),
        }

    # ----------------------------
    # Print comparison table
    # ----------------------------
    print(
        f"{'Scenario':<22} {'Avg Covers':<12} {'Peak Hr':<12} {'Day Total':<12} {'vs Normal'}"
    )
    print("-" * 80)

    normal_avg = results["normal_weekday"]["avg"]

    rows = []
    for name in scenarios:
        avg = results[name]["avg"]
        peak = results[name]["peak_hour_pred"]
        total_day = results[name]["total_day_pred"]
        diff = avg - normal_avg
        direction = "↑" if diff > 0 else "↓"
        print(f"{name:<22} {avg:<12.1f} {peak:<12.1f} {total_day:<12.1f} {direction} {diff:+.1f}")

        rows.append({
            "scenario": name,
            "avg_predicted_covers": round(avg, 2),
            "peak_hour_predicted_covers": round(peak, 2),
            "total_day_predicted_covers": round(total_day, 2),
            "delta_vs_normal": round(diff, 2),
        })

    validation_df = pd.DataFrame(rows)

    # ----------------------------
    # Rule-based sanity checks
    # ----------------------------
    print("\n" + "-" * 80)
    print("SANITY CHECKS")
    print("-" * 80)

    checks = []

    def add_check(label, condition, actual_text):
        status = "PASS" if condition else "FAIL"
        checks.append((label, status, actual_text))
        print(f"{status:<6} {label:<45} {actual_text}")

    add_check(
        "Weekend should exceed normal weekday",
        results["weekend"]["avg"] > results["normal_weekday"]["avg"],
        f"{results['weekend']['avg']:.1f} vs {results['normal_weekday']['avg']:.1f}"
    )

    add_check(
        "Holiday weekday should exceed normal weekday",
        results["holiday_weekday"]["avg"] > results["normal_weekday"]["avg"],
        f"{results['holiday_weekday']['avg']:.1f} vs {results['normal_weekday']['avg']:.1f}"
    )

    add_check(
        "Holiday weekend should exceed weekend",
        results["holiday_weekend"]["avg"] > results["weekend"]["avg"],
        f"{results['holiday_weekend']['avg']:.1f} vs {results['weekend']['avg']:.1f}"
    )

    add_check(
        "Promotion weekday should exceed normal weekday",
        results["promotion_weekday"]["avg"] >= results["normal_weekday"]["avg"],
        f"{results['promotion_weekday']['avg']:.1f} vs {results['normal_weekday']['avg']:.1f}"
    )

    add_check(
        "Event weekday should exceed normal weekday",
        results["event_weekday"]["avg"] >= results["normal_weekday"]["avg"],
        f"{results['event_weekday']['avg']:.1f} vs {results['normal_weekday']['avg']:.1f}"
    )

    add_check(
        "Event weekend should exceed weekend",
        results["event_weekend"]["avg"] >= results["weekend"]["avg"],
        f"{results['event_weekend']['avg']:.1f} vs {results['weekend']['avg']:.1f}"
    )

    # Rain is restaurant-dependent: snack/tea places may rise slightly in rain.
    # So we only check it is not absurdly different.
    rain_delta = abs(results["heavy_rain_weekday"]["avg"] - results["normal_weekday"]["avg"])
    add_check(
        "Heavy rain weekday should stay within reasonable band",
        rain_delta <= 6.0,
        f"delta = {rain_delta:.1f}"
    )

    hot_delta = results["hot_weekday"]["avg"] - results["normal_weekday"]["avg"]
    add_check(
        "Hot weekday should not exceed normal by too much",
        hot_delta <= 3.0,
        f"delta = {hot_delta:.1f}"
    )

    checks_df = pd.DataFrame(checks, columns=["check", "status", "details"])

    return results, validation_df, checks_df


# Example usage structure (you'd call this with your actual data):
_covers_df = covers_df.copy()

forecaster, train_df, test_df, pred, q10, q50, q90, metrics = run_single_xgb_forecast(_covers_df)

scenario_results, scenario_summary_df, scenario_checks_df = test_scenarios(
    forecaster=forecaster,
    covers_df=covers_df,
    train_end_ts=train_df["timestamp"].max()
)

print("\nScenario summary:")
print(scenario_summary_df)

print("\nScenario checks:")
print(scenario_checks_df)



SINGLE XGBOOST FORECAST RESULTS
Features used: 78
Train rows: 3795
Test rows: 105
MAE : 3.193
RMSE: 4.192
WAPE: 0.100

SCENARIO TESTING - SYSTEM VALIDATION

Scenario               Avg Covers   Peak Hr      Day Total    vs Normal
--------------------------------------------------------------------------------
normal_weekday         27.6         37.1         414.4        ↓ +0.0
heavy_rain_weekday     28.0         41.6         420.6        ↑ +0.4
hot_weekday            26.2         34.1         393.1        ↓ -1.4
weekend                32.7         39.8         489.8        ↑ +5.0
holiday_weekday        40.2         51.4         602.7        ↑ +12.6
holiday_weekend        43.2         53.1         647.3        ↑ +15.5
promotion_weekday      37.7         47.4         565.8        ↑ +10.1
event_weekday          43.5         52.5         652.6        ↑ +15.9
event_weekend          46.9         54.0         704.0        ↑ +19.3
rain_weekend           33.1         44.8         495.8        ↑

In [19]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd


def get_service_period(hour):
    if hour < 8:
        return "pre_service"
    elif 8 <= hour <= 11:
        return "breakfast"
    elif 12 <= hour <= 15:
        return "lunch"
    elif 16 <= hour <= 18:
        return "afternoon"
    elif 19 <= hour <= 22:
        return "dinner"
    else:
        return "late_night"


@dataclass
class PlannerConfig:
    data_dir: Path = Path("V2Data")
    prep_buffer_multiplier: float = 1.10
    covers_buffer_multiplier: float = 1.05


class StaffPlanner:
    """
    Convert forecasted covers into staffing requirements by role and station.

    The planner uses:
    - cover-based staffing rules for dining/floor roles such as server and manager
    - station workload derived from historical menu sales for production roles
    - a simple greedy shift builder to convert hourly demand into shift blocks
    """

    def __init__(self, config: Optional[PlannerConfig] = None):
        self.config = config or PlannerConfig()
        self.data_dir = Path(self.config.data_dir)

        self.historical_sales_df = pd.read_csv(
            self.data_dir / "historical_sales.csv", parse_dates=["timestamp"]
        )
        self.external_features_df = pd.read_csv(
            self.data_dir / "external_features.csv", parse_dates=["timestamp"]
        )
        self.historical_menu_sales_df = pd.read_csv(
            self.data_dir / "historical_menu_sales.csv", parse_dates=["timestamp"]
        )
        self.menu_item_master_df = pd.read_csv(self.data_dir / "menu_items_master.csv")
        self.staff_roles_df = pd.read_csv(self.data_dir / "staff_roles.csv")

        self._validate_core_schema()
        self.station_ratio_exact_df: Optional[pd.DataFrame] = None
        self.station_ratio_period_weekend_df: Optional[pd.DataFrame] = None
        self.station_ratio_period_df: Optional[pd.DataFrame] = None
        self.station_ratio_global_df: Optional[pd.DataFrame] = None

    def _validate_core_schema(self) -> None:
        missing_role_cols = {
            "role",
            "station",
            "capacity_unit",
            "capacity_per_hour",
            "min_staff_per_shift",
            "max_staff_per_shift",
            "shift_length_hours",
            "hourly_cost",
        } - set(self.staff_roles_df.columns)
        if missing_role_cols:
            raise ValueError(f"staff_roles.csv is missing columns: {sorted(missing_role_cols)}")

        if "covers" not in self.historical_sales_df.columns:
            raise ValueError("historical_sales.csv must contain a covers column")

        if "menu_item_id" not in self.historical_menu_sales_df.columns:
            raise ValueError("historical_menu_sales.csv must contain menu_item_id")

    def _prepare_context(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df["hour"] = df["timestamp"].dt.hour
        df["day_of_week"] = df["timestamp"].dt.dayofweek
        df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
        df["service_period"] = df["hour"].apply(get_service_period)

        if "rain_mm" in df.columns:
            df["heavy_rain_flag"] = (df["rain_mm"] >= 10).astype(int)
        else:
            df["rain_mm"] = 0.0
            df["heavy_rain_flag"] = 0

        for col in ["holiday_flag", "event_flag"]:
            if col not in df.columns:
                df[col] = 0

        return df

    def fit_station_workload_model(self) -> pd.DataFrame:

        menu_sales_df = self.historical_menu_sales_df.copy()
        menu_sales_df["timestamp"] = pd.to_datetime(menu_sales_df["timestamp"])

        active_menu_df = self.menu_item_master_df.copy()
        if "is_active" in active_menu_df.columns:
            active_menu_df = active_menu_df[active_menu_df["is_active"] == 1].copy()

        menu_station_df = menu_sales_df.merge(
            active_menu_df[["menu_item_id", "station", "prep_minutes"]],
            on="menu_item_id",
            how="left",
        )
        menu_station_df["prep_minutes"] = menu_station_df["prep_minutes"].fillna(0)
        menu_station_df["station_workload_minutes"] = (
            menu_station_df["qty_sold"] * menu_station_df["prep_minutes"]
        )

        # So for each hour, we get something like:
            # beverage: 85 prep minutes
            # light_kitchen: 120 prep minutes
            # main_kitchen: 160 prep minutes
        station_workload_df = (
            menu_station_df.groupby(["timestamp", "station"], as_index=False)[
                "station_workload_minutes"
            ]
            .sum()
            .sort_values(["timestamp", "station"])
        )

        context_df = self.historical_sales_df[["timestamp", "covers"]].merge(
            self.external_features_df, on="timestamp", how="left"
        )
        context_df = self._prepare_context(context_df)

        workload_history_df = station_workload_df.merge(
            context_df[
                [
                    "timestamp",
                    "covers",
                    "service_period",
                    "is_weekend",
                    "holiday_flag",
                    "event_flag",
                    "heavy_rain_flag",
                ]
            ],
            on="timestamp",
            how="left",
        )

        workload_history_df = workload_history_df[workload_history_df["covers"] > 0].copy()
        workload_history_df["minutes_per_cover"] = (
            workload_history_df["station_workload_minutes"] / workload_history_df["covers"]
        )

        self.station_ratio_exact_df = (
            workload_history_df.groupby(
                [
                    "station",
                    "service_period",
                    "is_weekend",
                    "holiday_flag",
                    "event_flag",
                    "heavy_rain_flag",
                ],
                as_index=False,
            )["minutes_per_cover"]
            .median()
            .rename(columns={"minutes_per_cover": "minutes_per_cover_exact"})
        )

        self.station_ratio_period_weekend_df = (
            workload_history_df.groupby(
                ["station", "service_period", "is_weekend"], as_index=False
            )["minutes_per_cover"]
            .median()
            .rename(columns={"minutes_per_cover": "minutes_per_cover_period_weekend"})
        )

        self.station_ratio_period_df = (
            workload_history_df.groupby(["station", "service_period"], as_index=False)[
                "minutes_per_cover"
            ]
            .median()
            .rename(columns={"minutes_per_cover": "minutes_per_cover_period"})
        )

        self.station_ratio_global_df = (
            workload_history_df.groupby("station", as_index=False)["minutes_per_cover"]
            .median()
            .rename(columns={"minutes_per_cover": "minutes_per_cover_global"})
        )

        return workload_history_df

    def estimate_station_workload(
        self,
        covers_forecast_df: pd.DataFrame,
        external_features_df: Optional[pd.DataFrame] = None,
    ) -> pd.DataFrame:
        
        if self.station_ratio_exact_df is None:
            self.fit_station_workload_model()

        forecast_df = covers_forecast_df.copy()
        forecast_df["timestamp"] = pd.to_datetime(forecast_df["timestamp"])

        if "predicted_covers" not in forecast_df.columns:
            raise ValueError("covers_forecast_df must contain a predicted_covers column")

        if external_features_df is None:
            external_features_df = self.external_features_df.copy()
        else:
            external_features_df = external_features_df.copy()

        external_features_df["timestamp"] = pd.to_datetime(external_features_df["timestamp"])

        forecast_df = forecast_df.merge(external_features_df, on="timestamp", how="left")
        forecast_df = self._prepare_context(forecast_df)

        stations_df = self.station_ratio_global_df[["station"]].copy()
        station_forecast_df = forecast_df.merge(stations_df, how="cross")

        station_forecast_df = station_forecast_df.merge(
            self.station_ratio_exact_df,
            on=[
                "station",
                "service_period",
                "is_weekend",
                "holiday_flag",
                "event_flag",
                "heavy_rain_flag",
            ],
            how="left",
        )
        station_forecast_df = station_forecast_df.merge(
            self.station_ratio_period_weekend_df,
            on=["station", "service_period", "is_weekend"],
            how="left",
        )
        station_forecast_df = station_forecast_df.merge(
            self.station_ratio_period_df,
            on=["station", "service_period"],
            how="left",
        )
        station_forecast_df = station_forecast_df.merge(
            self.station_ratio_global_df,
            on="station",
            how="left",
        )

        station_forecast_df["minutes_per_cover"] = (
            station_forecast_df["minutes_per_cover_exact"]
            .fillna(station_forecast_df["minutes_per_cover_period_weekend"])
            .fillna(station_forecast_df["minutes_per_cover_period"])
            .fillna(station_forecast_df["minutes_per_cover_global"])
            .fillna(0.0)
        )

        station_forecast_df["predicted_station_workload_minutes"] = (
            station_forecast_df["predicted_covers"]
            * station_forecast_df["minutes_per_cover"]
            * self.config.prep_buffer_multiplier
        )

        return station_forecast_df[
            [
                "timestamp",
                "station",
                "predicted_covers",
                "service_period",
                "is_weekend",
                "holiday_flag",
                "event_flag",
                "heavy_rain_flag",
                "minutes_per_cover",
                "predicted_station_workload_minutes",
            ]
        ].sort_values(["timestamp", "station"])

    def plan_hourly_staff(
        self,
        covers_forecast_df: pd.DataFrame,
        external_features_df: Optional[pd.DataFrame] = None,
    ) -> pd.DataFrame:
        
        station_workload_df = self.estimate_station_workload(
            covers_forecast_df=covers_forecast_df,
            external_features_df=external_features_df,
        )

        cover_roles_df = self.staff_roles_df[
            self.staff_roles_df["capacity_unit"] == "covers"
        ].copy()
        prep_roles_df = self.staff_roles_df[
            self.staff_roles_df["capacity_unit"] == "prep_minutes"
        ].copy()

        cover_plan_frames = []
        if not cover_roles_df.empty:
            base_forecast_df = covers_forecast_df.copy()
            base_forecast_df["timestamp"] = pd.to_datetime(base_forecast_df["timestamp"])
            for _, role_row in cover_roles_df.iterrows():
                role_df = base_forecast_df.copy()
                role_df["role"] = role_row["role"]
                role_df["station"] = role_row["station"]
                role_df["capacity_unit"] = role_row["capacity_unit"]
                role_df["capacity_per_hour"] = role_row["capacity_per_hour"]
                role_df["min_staff_per_shift"] = role_row["min_staff_per_shift"]
                role_df["max_staff_per_shift"] = role_row["max_staff_per_shift"]
                role_df["shift_length_hours"] = role_row["shift_length_hours"]
                role_df["hourly_cost"] = role_row["hourly_cost"]
                role_df["required_load"] = (
                    role_df["predicted_covers"] * self.config.covers_buffer_multiplier
                )
                cover_plan_frames.append(role_df)

        cover_plan_df = (
            pd.concat(cover_plan_frames, ignore_index=True)
            if cover_plan_frames
            else pd.DataFrame()
        )

        prep_plan_df = station_workload_df.merge(
            prep_roles_df,
            on="station",
            how="inner",
        )
        if not prep_plan_df.empty:
            prep_plan_df["required_load"] = prep_plan_df["predicted_station_workload_minutes"]

        hourly_plan_df = pd.concat(
            [df for df in [cover_plan_df, prep_plan_df] if not df.empty],
            ignore_index=True,
        )

        hourly_plan_df["required_staff"] = np.ceil(
            hourly_plan_df["required_load"] / hourly_plan_df["capacity_per_hour"]
        ).astype(int)

        hourly_plan_df["required_staff"] = hourly_plan_df["required_staff"].clip(
            lower=hourly_plan_df["min_staff_per_shift"],
            upper=hourly_plan_df["max_staff_per_shift"],
        )
        
        hourly_plan_df["estimated_hourly_cost"] = (
            hourly_plan_df["required_staff"] * hourly_plan_df["hourly_cost"]
        ).round(2)

        output_cols = [
            "timestamp",
            "role",
            "station",
            "capacity_unit",
            "capacity_per_hour",
            "required_load",
            "required_staff",
            "shift_length_hours",
            "hourly_cost",
            "estimated_hourly_cost",
        ]
        return hourly_plan_df[output_cols].sort_values(["timestamp", "role", "station"])

    def build_shift_schedule(self, hourly_plan_df: pd.DataFrame) -> pd.DataFrame:
        if hourly_plan_df.empty:
            return pd.DataFrame(
                columns=[
                    "date",
                    "role",
                    "station",
                    "shift_start",
                    "shift_end",
                    "staff_count",
                    "shift_length_hours",
                    "hourly_cost",
                    "estimated_shift_cost",
                ]
            )

        hourly_plan_df = hourly_plan_df.copy()
        hourly_plan_df["timestamp"] = pd.to_datetime(hourly_plan_df["timestamp"])
        hourly_plan_df["date"] = hourly_plan_df["timestamp"].dt.date

        shift_rows = []

        for (date_value, role, station), group_df in hourly_plan_df.groupby(
            ["date", "role", "station"], sort=True
        ):
            group_df = group_df.sort_values("timestamp").reset_index(drop=True)
            shift_length_hours = int(group_df["shift_length_hours"].iloc[0])
            hourly_cost = float(group_df["hourly_cost"].iloc[0])
            last_timestamp = group_df["timestamp"].max()
            active_shift_end_times: list[pd.Timestamp] = []

            for _, row in group_df.iterrows():
                current_ts = row["timestamp"]
                active_shift_end_times = [
                    end_ts for end_ts in active_shift_end_times if end_ts > current_ts
                ]
                active_count = len(active_shift_end_times)
                required_count = int(row["required_staff"])

                if active_count >= required_count:
                    continue

                new_shifts_needed = required_count - active_count
                for _ in range(new_shifts_needed):
                    natural_end = current_ts + pd.Timedelta(hours=shift_length_hours)
                    clipped_end = min(natural_end, last_timestamp + pd.Timedelta(hours=1))
                    active_shift_end_times.append(clipped_end)
                    shift_rows.append(
                        {
                            "date": date_value,
                            "role": role,
                            "station": station,
                            "shift_start": current_ts,
                            "shift_end": clipped_end,
                            "staff_count": 1,
                            "shift_length_hours": (clipped_end - current_ts).total_seconds()
                            / 3600,
                            "hourly_cost": hourly_cost,
                            "estimated_shift_cost": round(
                                ((clipped_end - current_ts).total_seconds() / 3600)
                                * hourly_cost,
                                2,
                            ),
                        }
                    )

        shift_schedule_df = pd.DataFrame(shift_rows)
        if shift_schedule_df.empty:
            return shift_schedule_df

        return (
            shift_schedule_df.groupby(
                [
                    "date",
                    "role",
                    "station",
                    "shift_start",
                    "shift_end",
                    "shift_length_hours",
                    "hourly_cost",
                ],
                as_index=False,
            )["staff_count"]
            .sum()
            .assign(
                estimated_shift_cost=lambda df: (
                    df["staff_count"] * df["shift_length_hours"] * df["hourly_cost"]
                ).round(2)
            )
            .sort_values(["date", "shift_start", "role", "station"])
        )


def build_staffing_plan_from_forecast(
    covers_forecast_df: pd.DataFrame,
    data_dir: str | Path = "V2Data",
    external_features_df: Optional[pd.DataFrame] = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    
    planner = StaffPlanner(PlannerConfig(data_dir=Path(data_dir)))
    
    hourly_plan_df = planner.plan_hourly_staff(
        covers_forecast_df=covers_forecast_df,
        external_features_df=external_features_df,
    )
    
    shift_schedule_df = planner.build_shift_schedule(hourly_plan_df)
    
    return hourly_plan_df, shift_schedule_df


In [ ]:
import pandas as pd

covers_df = covers_df.sort_values("timestamp").reset_index(drop=True)

# Run covers forecast
forecaster, train_df, test_df, pred, q10, q50, q90, metrics = run_single_xgb_forecast(
    covers_df,
    holdout_hours=7 * 15
)

print("\nForecast metrics")
print(metrics)

# Build the forecast dataframe expected by the staff planner
covers_forecast_df = test_df[["timestamp"]].copy()
covers_forecast_df["predicted_covers"] = pred
covers_forecast_df["p10_covers"] = q10
covers_forecast_df["p50_covers"] = q50
covers_forecast_df["p90_covers"] = q90

# Optional: pass matching external features for those timestamps
future_external_df = external_features_df[
    external_features_df["timestamp"].isin(covers_forecast_df["timestamp"])
].copy()

# Build staffing outputs
hourly_plan_df, shift_schedule_df = build_staffing_plan_from_forecast(
    covers_forecast_df=covers_forecast_df,
    data_dir="V2Data",
    external_features_df=future_external_df,
)

print("\nHourly staff plan")
print(hourly_plan_df.head(20))

print("\nShift schedule")
print(shift_schedule_df.head(20))




SINGLE XGBOOST FORECAST RESULTS
Features used: 78
Train rows: 3795
Test rows: 105
MAE : 3.193
RMSE: 4.192
WAPE: 0.100

Forecast metrics
{'mae': 3.1932668685913086, 'rmse': np.float64(4.19222255127678), 'wape': np.float64(0.10026705521715885)}


In [ ]:
# Test staff
def print_section(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

service_hours_per_day = historical_sales_df["timestamp"].dt.hour.nunique()
holdout_hours = 14 * service_hours_per_day

(
    forecaster,
    train_df,
    test_df,
    pred,
    q10,
    q50,
    q90,
    metrics,
) = run_single_xgb_forecast(covers_df, holdout_hours=holdout_hours)

covers_forecast_df = test_df[["timestamp", "covers"]].copy()
covers_forecast_df = covers_forecast_df.rename(columns={"covers": "actual_covers"})
covers_forecast_df["predicted_covers"] = pred.round(2)
covers_forecast_df["abs_error"] = (
    covers_forecast_df["actual_covers"] - covers_forecast_df["predicted_covers"]
).abs().round(2)

staffing_input_df = covers_forecast_df[["timestamp", "predicted_covers"]]

future_external_df = external_features_df[
    external_features_df["timestamp"].isin(staffing_input_df["timestamp"])
].copy()

planner = StaffPlanner(PlannerConfig())
station_workload_df = planner.estimate_station_workload(
    covers_forecast_df=staffing_input_df,
    external_features_df=future_external_df,
)
hourly_plan_df = planner.plan_hourly_staff(
    covers_forecast_df=staffing_input_df,
    external_features_df=future_external_df,
)
shift_schedule_df = planner.build_shift_schedule(hourly_plan_df)

staffing_summary_df = (
    hourly_plan_df.groupby(["role", "station"], as_index=False)
    .agg(
        avg_staff=("required_staff", "mean"),
        peak_staff=("required_staff", "max"),
        total_hourly_cost=("estimated_hourly_cost", "sum"),
    )
    .sort_values(["role", "station"])
)
staffing_summary_df["avg_staff"] = staffing_summary_df["avg_staff"].round(2)
staffing_summary_df["total_hourly_cost"] = staffing_summary_df[
    "total_hourly_cost"
].round(2)

busiest_hours_df = (
    hourly_plan_df.groupby("timestamp", as_index=False)
    .agg(
        total_staff=("required_staff", "sum"),
        total_hourly_cost=("estimated_hourly_cost", "sum"),
    )
    .sort_values(["total_staff", "total_hourly_cost"], ascending=[False, False])
)

print_section("Forecast Metrics")
print(pd.DataFrame([metrics]).to_string(index=False))

print_section("Forecast Sample")
print(covers_forecast_df.head(15).to_string(index=False))

print_section("Station Workload Sample")
print(station_workload_df.head(20).to_string(index=False))

print_section("Hourly Staff Plan Sample")
print(hourly_plan_df.head(20).to_string(index=False))

print_section("Staffing Summary By Role And Station")
print(staffing_summary_df.to_string(index=False))

print_section("Top 10 Busiest Hours")
print(busiest_hours_df.head(10).to_string(index=False))

print_section("Shift Schedule Sample")
print(shift_schedule_df.head(20).to_string(index=False))



SINGLE XGBOOST FORECAST RESULTS
Features used: 78
Train rows: 3690
Test rows: 210
MAE : 3.174
RMSE: 4.102
WAPE: 0.103

Forecast Metrics
     mae     rmse     wape
3.173852 4.102432 0.102809

Forecast Sample
          timestamp  actual_covers  predicted_covers  p10_covers  p50_covers  p90_covers  abs_error
2025-12-18 08:00:00             22         21.959999   18.059999   20.780001   25.830000       0.04
2025-12-18 09:00:00             15         14.590000   11.300000   14.650000   22.389999       0.41
2025-12-18 10:00:00             16         14.300000   10.870000   15.290000   19.770000       1.70
2025-12-18 11:00:00             18         13.570000   10.770000   14.660000   19.480000       4.43
2025-12-18 12:00:00             25         31.370001   27.129999   32.570000   37.200001       6.37
2025-12-18 13:00:00             35         32.139999   27.340000   33.419998   39.480000       2.86
2025-12-18 14:00:00             34         31.559999   26.230000   32.230000   39.209999    